# EEG — BIOT / IIIC **full fine-tune** (unfreeze the encoder), GPU

This notebook is the *full-data, full-fine-tune* evolution of `tools/colab_eeg.ipynb`.
The deployed EEG head was trained with the **encoder frozen** on a small CPU subset and
plateaus at **balanced accuracy 0.278** (κ ≈ 0.15). Here we **unfreeze the BIOT encoder**
and train end-to-end on a larger HMS subset on a GPU.

### Honest target — read before your defence
- **Baseline:** 0.278 balanced accuracy (frozen-encoder linear probe, 6-class IIIC).
- **Realistic target:** **0.45 – 0.55 balanced accuracy** — i.e. roughly BIOT's *published*
  IIIC level (~0.5). That is the ceiling we aim for, not 90 %.
- **Why 90 % is impossible here:** IIIC (Ictal–Interictal–Injury Continuum, 6 classes) is a
  genuinely ambiguous task — *expert neurologists disagree with each other* on these labels.
  The field scores it with **balanced accuracy** and **Cohen's κ** against *consensus* labels
  precisely because inter-rater agreement is far below 90 %. BIOT's own published full-data
  result is ≈ **0.5** balanced accuracy and that is considered strong. A model claiming 90 %
  on IIIC would be out-performing the experts who *made* the labels — a leakage red flag, not
  a result. The defensible claim after this notebook: *"the full GPU fine-tune lifts our
  screening head from 0.28 toward ≈ 0.5 balanced accuracy, matching the encoder authors'
  published level."*

### What you need first (one-time)
1. **Runtime → Change runtime type → T4 GPU.**
2. Build the lean repo zip on your PC and upload it to your **Google Drive root** (`My Drive/`):
   `python "Colab PFE/make_lean_zip.py"` → upload the produced `medical-platform.zip`.
   (It already contains the bundled BIOT encoder `EEG-PREST-16-channels.ckpt`.)
3. A **Kaggle API token** (`kaggle.json`) and a one-time acceptance of the
   [HMS competition rules](https://www.kaggle.com/competitions/hms-harmful-brain-activity-classification/rules).

### Colab timeout note (important)
A free **T4 session is capped at ~12 h** (and idles out sooner if you close the tab). This whole
run — rate-limited HMS download + full fine-tune + eval — fits comfortably for the default
`N_EEGS = 5000`. The **download cell is resumable**: re-running it skips parquet files already on
disk (and `train.csv` is cached), so a dropped session just continues. If you raise `N_EEGS`
toward the full ~17 k EEGs, do it across sessions and point `HMS_DIR` at a Drive folder so the
downloaded data survives a disconnect (slower I/O, but persistent).

Run the cells **top to bottom**. The final cell prints a `=== RESULTS ===` block — paste it into
`Colab PFE/RESULTS_TEMPLATE.md` and hand it back to Claude for local re-validation.

## 1 — Confirm the GPU
If this prints `NO GPU`, set **Runtime → Change runtime type → T4 GPU** and re-run. The
full fine-tune is impractically slow on CPU.

In [ ]:
import torch
print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')

## 2 — Mount Drive and unpack the repo
Unzips `My Drive/medical-platform.zip` to `/content/medical-platform`. The flatten check (from
`tools/COLAB.md`) handles the case where the zip nested an extra top folder.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!rm -rf /content/medical-platform
!mkdir -p /content/medical-platform
!unzip -q "/content/drive/My Drive/medical-platform.zip" -d /content/medical-platform
import os
root = "/content/medical-platform"
if not os.path.exists(f"{root}/tools") and os.path.exists(f"{root}/medical-platform/tools"):
    root = f"{root}/medical-platform"
print("repo root:", root)
%cd $root

## 3 — Install the extra deps
Colab already ships torch / numpy / pandas. BIOT needs `linear-attention-transformer`; the HMS
parquet reader needs `pyarrow`. (The full fine-tune needs nothing beyond what BIOT itself imports.)

In [ ]:
!pip -q install linear-attention-transformer pyarrow

## 4 — Run configuration (edit here if you want)
`N_EEGS` is the only knob most people touch.

**Why this default and not "ALL".** HMS has ~**17,089** unique labelled EEGs. The repo's
downloader (`train_eeg_head.download_subset`, also reachable as `train_eeg_head.py --download N`)
fetches up to **N** of them **one file at a time** through the rate-limited Kaggle API (a 0.25 s
pause between files plus exponential back-off on HTTP 429). Pulling all ~17 k in one session would
take hours of throttled transfer and risks the 12 h T4 cap — so there is no practical "ALL" switch.
The default **`N_EEGS = 5000`** is the *largest comfortably-practical* subset for one session
(roughly double `tools/colab_eeg.ipynb`'s 2500). `LIMIT` caps the labelled windows actually used,
balanced across the 6 classes, so RAM stays bounded. To approach full data, raise `N_EEGS`, point
`HMS_DIR` at a Drive folder, and let the resumable download cell run across sessions.

In [ ]:
import time

# --- main knobs ---
N_EEGS     = 5000          # how many unique HMS EEGs to download (see markdown above)
LIMIT      = 20000         # max labelled windows used, balanced across the 6 IIIC classes
EPOCHS     = 10            # --unfreeze wants FEW epochs (~6-12): the encoder is already pretrained
BATCH_SIZE = 32            # raw (16,2000) segments flow through the encoder; drop to 16 on OOM
ENC_LR     = "1e-5"        # encoder LR (small — fine-tune, don't wreck the pretraining)
HEAD_LR    = "1e-3"        # classification-head LR
SEED       = 0             # fixes BOTH the balanced sampling and the patient-disjoint split

# --- paths ---
HMS_DIR  = "data/hms"            # local (fast). For cross-session persistence use a Drive path, e.g.
                                 #   HMS_DIR = "/content/drive/My Drive/hms_data"
OUT_CKPT = "/content/biot_iiic.pt"   # full BIOTClassifier state_dict (encoder + trained head)

T_START = time.time()
print("config:", dict(N_EEGS=N_EEGS, LIMIT=LIMIT, EPOCHS=EPOCHS, BATCH_SIZE=BATCH_SIZE,
                       ENC_LR=ENC_LR, HEAD_LR=HEAD_LR, SEED=SEED, HMS_DIR=HMS_DIR, OUT_CKPT=OUT_CKPT))

## 5 — Kaggle credentials
First **accept the HMS rules once** (needed regardless of token type):
https://www.kaggle.com/competitions/hms-harmful-brain-activity-classification/rules

Two ways to authenticate — the cell below supports both:
- **New-style token (recommended)** — looks like `KGAT_…` (kaggle.com → *Settings* → *API* → create token). Just paste it when prompted; **no username, no kaggle.json needed**. The repo's downloader sends it as a Bearer token.
- **Legacy `kaggle.json`** — press Enter at the prompt instead, then upload the file.

The token is read with `getpass`, so it is never displayed or saved in the notebook.

In [ ]:
from getpass import getpass
import os

tok = getpass("Paste your Kaggle token (KGAT_...), or press Enter to upload kaggle.json instead: ").strip()
if tok:
    os.environ["KAGGLE_API_TOKEN"] = tok
    print("KAGGLE_API_TOKEN set (Bearer auth - no username needed).")
else:
    from google.colab import files
    import json
    up = files.upload()                         # pick your kaggle.json
    k = json.load(open("kaggle.json"))
    os.environ["KAGGLE_USERNAME"] = k["username"]
    os.environ["KAGGLE_KEY"] = k["key"]
    print("legacy kaggle.json creds set for", k["username"])

## 6 — Download the HMS subset (resumable)
Calls the repo's own `download_subset` (the same code behind `train_eeg_head.py --download`). It
pulls `train.csv` once, then a class-balanced set of `N_EEGS` per-EEG parquet files into
`HMS_DIR/train_eegs/`. **Re-running is safe and resumable** — files already on disk are skipped, so
a dropped session just continues from here. Expect ~20–60 min for 5000 EEGs (rate-limited).

In [ ]:
import time, sys
from pathlib import Path
sys.path.insert(0, "tools")
from train_eeg_head import download_subset    # importing is light (no torch at module level)

t_dl = time.time()
download_subset(Path(HMS_DIR), N_EEGS)
print("download wall-clock: {:.0f} min".format((time.time() - t_dl) / 60))

## 7 — Full fine-tune (unfreeze the encoder)
Runs `tools/train_eeg_head.py --unfreeze`: it builds `BIOTClassifier`, loads the bundled BIOT
encoder, **unfreezes it**, and trains encoder+head end-to-end with a class-balanced loss. It saves
the best-val-balanced-acc state to `OUT_CKPT` as a **full** `BIOTClassifier` state_dict — which
`model_loader.get_eeg_model()` loads as-is (drop it at
`backend/models_weights/biot/biot_iiic.pt` locally, no code change).

Watch the per-epoch `val balanced-acc`. If it is still climbing at the last epoch, raise `EPOCHS`.
On out-of-memory, lower `BATCH_SIZE` to 16 or `LIMIT` to 10000 in the config cell and re-run.

In [ ]:
import time
t_train = time.time()
!python tools/train_eeg_head.py --hms-dir "{HMS_DIR}" --limit {LIMIT} --unfreeze --epochs {EPOCHS} --batch-size {BATCH_SIZE} --encoder-lr {ENC_LR} --lr {HEAD_LR} --seed {SEED} --out "{OUT_CKPT}"
t_train_min = (time.time() - t_train) / 60
print("training wall-clock: {:.0f} min".format(t_train_min))

## 8 — Evaluate on the patient-disjoint held-out split
Runs `tools/eval_eeg.py` with the **same `--limit` and `--seed`** as training, so it reproduces the
identical patient-disjoint split (no leakage). The printout (balanced-acc, κ, macro/weighted F1, KL,
per-class table, 6×6 confusion matrix) is also saved to `eeg_eval_report.txt`.

In [ ]:
!python tools/eval_eeg.py --hms-dir "{HMS_DIR}" --weights "{OUT_CKPT}" --limit {LIMIT} --seed {SEED} 2>&1 | tee eeg_eval_report.txt

## 9 — Build a metrics `.json` and copy outputs to Drive
Recomputes the headline metrics by **reusing `eval_eeg.py`'s own functions** (so the numbers match
cell 8 exactly), writes `eeg_metrics.json`, and copies the checkpoint + json + text report to
`My Drive/colab_pfe_outputs/eeg/`. This runs inference once more on the (small) held-out split.

In [ ]:
import os, sys, json, shutil
import numpy as np
import torch

sys.path.insert(0, "tools")
sys.path.insert(0, "backend")
from apps.inference.biot import BIOTClassifier
from apps.inference.eeg_preprocess import IIIC_CLASSES
from eeg_hms import iter_segments, load_index, patient_split
from eval_eeg import _confusion, _metrics_from_confusion, _kl_divergence

device = "cuda" if torch.cuda.is_available() else "cpu"
model = BIOTClassifier(n_classes=6, n_channels=16, n_fft=200, hop_length=100)
state = torch.load(OUT_CKPT, map_location=device)
state = state.get("state_dict", state) if isinstance(state, dict) else state
model.load_state_dict(state)
model.eval().to(device)

# same patient-disjoint held-out split the trainer/eval used (same limit + seed)
samples = load_index(HMS_DIR, limit=LIMIT, balanced=True, seed=SEED)
_, test_s = patient_split(samples, test_frac=0.2, seed=SEED)

y_true, y_pred, votes, probs = [], [], [], []
with torch.no_grad():
    for s, seg in iter_segments(HMS_DIR, test_s):
        x = torch.from_numpy(seg[None].astype(np.float32)).to(device)
        p = torch.softmax(model(x), dim=1).cpu().numpy()[0]
        y_true.append(s["label"])
        y_pred.append(int(p.argmax()))
        votes.append(s["votes"])
        probs.append(p)

c = _confusion(np.array(y_true), np.array(y_pred))
m = _metrics_from_confusion(c)
kl = _kl_divergence(votes, probs)

metrics = {
    "run": "EEG BIOT/IIIC full fine-tune (unfreeze)",
    "baseline_balanced_acc": 0.278,
    "balanced_acc": round(float(m["balanced_acc"]), 4),
    "cohen_kappa": round(float(m["kappa"]), 4),
    "macro_f1": round(float(m["macro_f1"]), 4),
    "weighted_f1": round(float(m["weighted_f1"]), 4),
    "raw_accuracy": round(float(m["accuracy"]), 4),
    "kl_divergence": round(float(kl), 4),
    "n_test_windows": int(len(y_true)),
    "classes": list(IIIC_CLASSES),
    "per_class": {
        IIIC_CLASSES[i]: {
            "precision": round(float(m["precision"][i]), 4),
            "recall": round(float(m["recall"][i]), 4),
            "f1": round(float(m["f1"][i]), 4),
            "support": int(m["support"][i]),
        }
        for i in range(6)
    },
    "confusion_matrix": c.tolist(),
    "config": {
        "n_eegs": N_EEGS, "limit": LIMIT, "epochs": EPOCHS, "batch_size": BATCH_SIZE,
        "encoder_lr": ENC_LR, "head_lr": HEAD_LR, "seed": SEED,
    },
}

DRIVE_OUT = "/content/drive/My Drive/colab_pfe_outputs/eeg"
os.makedirs(DRIVE_OUT, exist_ok=True)
DRIVE_CKPT = os.path.join(DRIVE_OUT, "biot_iiic.pt")
DRIVE_JSON = os.path.join(DRIVE_OUT, "eeg_metrics.json")
DRIVE_REPORT = os.path.join(DRIVE_OUT, "eeg_eval_report.txt")
shutil.copy(OUT_CKPT, DRIVE_CKPT)
with open(DRIVE_JSON, "w") as f:
    json.dump(metrics, f, indent=2)
if os.path.exists("eeg_eval_report.txt"):
    shutil.copy("eeg_eval_report.txt", DRIVE_REPORT)
print("copied checkpoint + metrics + report to", DRIVE_OUT)

## 10 — RESULTS (paste this whole block back to Claude)

In [ ]:
import time
print("=== RESULTS - paste this back to Claude ===")
print("run: EEG BIOT/IIIC FULL fine-tune (unfreeze encoder)")
print("baseline (frozen-encoder probe): balanced-acc 0.278  |  realistic target 0.45-0.55  |  BIOT published ~0.5")
print("config: N_EEGS={} LIMIT={} EPOCHS={} batch={} enc_lr={} head_lr={} seed={}".format(
    N_EEGS, LIMIT, EPOCHS, BATCH_SIZE, ENC_LR, HEAD_LR, SEED))
print("test windows (patient-disjoint): {}".format(metrics["n_test_windows"]))
print("balanced_acc : {:.3f}".format(metrics["balanced_acc"]))
print("cohen_kappa  : {:.3f}".format(metrics["cohen_kappa"]))
print("macro_f1     : {:.3f}".format(metrics["macro_f1"]))
print("weighted_f1  : {:.3f}".format(metrics["weighted_f1"]))
print("kl_divergence: {:.3f}  (true votes || predicted, lower is better)".format(metrics["kl_divergence"]))
print("raw_accuracy : {:.3f}  (misleading under imbalance)".format(metrics["raw_accuracy"]))
print("per-class F1:")
for name in metrics["classes"]:
    pc = metrics["per_class"][name]
    print("  {:<6} f1={:.3f} prec={:.3f} rec={:.3f} support={}".format(
        name, pc["f1"], pc["precision"], pc["recall"], pc["support"]))
print("confusion matrix (rows=true, cols=pred), class order {}:".format(metrics["classes"]))
for r in metrics["confusion_matrix"]:
    print("  {}".format(r))
print("files on Drive (My Drive/colab_pfe_outputs/eeg/):")
print("  checkpoint  : {}".format(DRIVE_CKPT))
print("  metrics json: {}".format(DRIVE_JSON))
print("  eval report : {}".format(DRIVE_REPORT))
print("place biot_iiic.pt at backend/models_weights/biot/biot_iiic.pt locally (no code change needed)")
print("runtime: training {:.0f} min  |  notebook total {:.0f} min".format(
    t_train_min, (time.time() - T_START) / 60))
print("=== END RESULTS ===")